# 01 · Exploración y limpieza
### Proyecto Olist · Python → MySQL → Tableau

---

## Contexto de negocio

**Olist** es un marketplace brasileño que conecta vendedores independientes con clientes en todo Brasil. Opera como intermediario: los vendedores publican sus productos en Olist, y Olist se encarga de la visibilidad, la logística y el soporte.

El dataset recoge **99.441 pedidos realizados entre 2016 y 2018**, con información de clientes, productos y valoraciones.

---

## Pregunta de negocio

> **¿Quién es el cliente de Olist, qué compra, cuándo lo compra y está satisfecho?**

Esa pregunta guía todo el proyecto. Solo analizamos lo que ayuda a responderla.

---

## Tablas utilizadas

| | Tabla | Pregunta que responde |
|---|---|---|
| ✅ | `customers` | ¿Dónde están los clientes? |
| ✅ | `orders` | ¿Cuándo compran? |
| ✅ | `order_items` | ¿A qué precio vendemos? |
| ✅ | `products` | ¿Qué categorías existen? |
| ✅ | `reviews` | ¿Están satisfechos? |
| ❌ | `payments` | Fuera del foco del dashboard |
| ❌ | `geolocation` | Redundante con `customers` |
| ❌ | `sellers` | Fuera del foco del análisis |
| ❌ | `product_category_name_translation` | No mezclar idiomas |

---

## Objetivo de este notebook

Limpiar las 5 tablas necesarias y guardar los CSV limpios en `output/`.  
El notebook 02 los cargará en MySQL.

---
# Librerías

In [1]:
import pandas as pd  # tablas de datos
import os            # rutas de archivos

---
# 1️⃣ Carga de datos

**Dataset:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

In [2]:
data_path   = "data"    # CSVs originales — nunca se modifican
output_path = "output"  # CSVs limpios — resultado de la limpieza

os.makedirs(output_path, exist_ok=True)  # crea output/ si no existe

customers_raw   = pd.read_csv(os.path.join(data_path, "olist_customers_dataset.csv")) # abre el CSV como tabla
orders_raw      = pd.read_csv(os.path.join(data_path, "olist_orders_dataset.csv"))
order_items_raw = pd.read_csv(os.path.join(data_path, "olist_order_items_dataset.csv"))
products_raw    = pd.read_csv(os.path.join(data_path, "olist_products_dataset.csv"))
reviews_raw     = pd.read_csv(os.path.join(data_path, "olist_order_reviews_dataset.csv"))

print("✅ Datos cargados correctamente")

✅ Datos cargados correctamente


---
# 2️⃣ Exploración y limpieza individual

Esquema aplicado a cada tabla: **dimensión → información general → inspección → limpieza → guardar**

## 2.1 CUSTOMERS

In [3]:
## 1️⃣ DIMENSIÓN
print(f"Filas: {customers_raw.shape[0]:,} | Columnas: {customers_raw.shape[1]}")

Filas: 99,441 | Columnas: 5


In [4]:
## 2️⃣ INFORMACIÓN GENERAL
customers_raw.info()
print("\nÚnicos:\n",   customers_raw.nunique())
print("\nNulos:\n",    customers_raw.isnull().any())
print("\nDuplicados:", customers_raw.duplicated().any())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB

Únicos:
 customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
dtype: int64

Nulos:
 customer_id                 False
customer_unique_id          False
customer_zip_code_prefix    False
customer_city               False
customer_state              False
dtype: bool

Duplicados: False


In [5]:
## 3️⃣ INSPECCIÓN
print("Primeras:\n",    customers_raw.head(3))
print("\nÚltimas:\n",   customers_raw.tail(3))

Primeras:
                         customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  

Últimas:
                             customer_id                customer_unique_id  \
99438  5e28dfe12db7fb50a4b2f691faecea5e  e9f50caf99f032f0bf3c55141f019d99   
99439  56b18e2166679b8a959d72dd06da27f9  73c2643a0a458b49f58cea58833b192e   
99440  274fa6071e5e17fe303b9748641082c8  84732c5050c01db9b23e19ba39899398   

       customer_zip_code_prefix customer_city customer_state  
99438                     60115     forta

In [6]:
## 4️⃣ LIMPIEZA
customers_clean = customers_raw.copy()

# 1) SELECCIÓN DE COLUMNAS
customers_clean = customers_clean[['customer_id', 'customer_state']]
print("✅ Columnas seleccionadas")

# 2) CORRECCIÓN DE TIPOS
# los IDs son etiquetas, no números ni categorías — su tipo debe ser string
customers_clean['customer_id'] = customers_clean['customer_id'].astype('string')
print("✅ Tipos corregidos")

# 3) NORMALIZACIÓN DE TEXTO
customers_clean.columns = customers_clean.columns.str.strip().str.lower().str.replace(' ', '_')
for col in customers_clean.select_dtypes(include=['object', 'string']).columns:
    customers_clean[col] = customers_clean[col].str.strip().str.lower().str.replace(' ', '_')
print("✅ Texto normalizado")

# 4) MAPEO DE ESTADOS
# siglas → nombres completos (sp → sao_paulo) para mejorar la legibilidad en el dashboard
state_mapping = {
    'ac': 'acre',             'al': 'alagoas',             'ap': 'amapa',
    'am': 'amazonas',         'ba': 'bahia',               'ce': 'ceara',
    'df': 'distrito_federal', 'es': 'espirito_santo',      'go': 'goias',
    'ma': 'maranhao',         'mt': 'mato_grosso',         'ms': 'mato_grosso_do_sul',
    'mg': 'minas_gerais',     'pa': 'para',                'pb': 'paraiba',
    'pr': 'parana',           'pe': 'pernambuco',          'pi': 'piaui',
    'rj': 'rio_de_janeiro',   'rn': 'rio_grande_do_norte', 'rs': 'rio_grande_do_sul',
    'ro': 'rondonia',         'rr': 'roraima',             'sc': 'santa_catarina',
    'sp': 'sao_paulo',        'se': 'sergipe',             'to': 'tocantins'
}
customers_clean['customer_state'] = customers_clean['customer_state'].replace(state_mapping)
print("✅ Estados mapeados")

✅ Columnas seleccionadas
✅ Tipos corregidos
✅ Texto normalizado
✅ Estados mapeados


In [7]:
customers_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customer_id     99441 non-null  string
 1   customer_state  99441 non-null  object
dtypes: object(1), string(1)
memory usage: 1.5+ MB


In [8]:
## 5️⃣ GUARDAR
customers_clean.to_csv(os.path.join(output_path, "customers_clean.csv"), index=False)
print("💾 customers_clean.csv guardado")

💾 customers_clean.csv guardado


## 2.2 ORDERS

In [9]:
## 1️⃣ DIMENSIÓN
print(f"Filas: {orders_raw.shape[0]:,} | Columnas: {orders_raw.shape[1]}")

Filas: 99,441 | Columnas: 8


In [10]:
## 2️⃣ INFORMACIÓN GENERAL
orders_raw.info()
print("\nÚnicos:\n",    orders_raw.nunique())
print("\nNulos (%):\n", orders_raw.isnull().mean() * 100)
print("\nDuplicados:",  orders_raw.duplicated().any())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB

Únicos:
 order_id                         99441
customer_id                      99441
order_status                         8
order_purchase_timestamp         98875
order_approved_at                90733
order_delivered_carrier_date     81018
order_delivered_cu

In [11]:
## 3️⃣ INSPECCIÓN
print("Primeras:\n",    orders_raw.head(3))
print("\nÚltimas:\n",   orders_raw.tail(3))

Primeras:
                            order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15:27:45   
2          2018-08-08 13:50:00           2018-08-17 18:06:29   

  order_estimated_delivery_date  
0           2017-10-18 00:00:00  
1           2018-08-13 00:00:00  
2           2018-09-04 00:00:00  

Últimas:
                                orde

In [12]:
## 4️⃣ LIMPIEZA
orders_clean = orders_raw.copy()

# 1) SELECCIÓN DE COLUMNAS
orders_clean = orders_clean[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']]
print("✅ Columnas seleccionadas")

# 2) FILTRO DE PEDIDOS
# solo pedidos entregados — los cancelados o en tránsito no tienen valoración del cliente
antes = len(orders_clean)
orders_clean = orders_clean[orders_clean['order_status'] == 'delivered']
print(f"✅ Pedidos no entregados descartados: {antes - len(orders_clean):,}")
orders_clean = orders_clean.drop(columns=['order_status'])  # ya no necesaria tras el filtro

# 3) CORRECCIÓN DE TIPOS
# los IDs son etiquetas, no números ni categorías — su tipo debe ser string
orders_clean['order_id']    = orders_clean['order_id'].astype('string')
orders_clean['customer_id'] = orders_clean['customer_id'].astype('string')
# pd.to_datetime() convierte el texto a fecha operable — necesario para el análisis temporal
orders_clean['order_purchase_timestamp'] = pd.to_datetime(orders_clean['order_purchase_timestamp'])
print("✅ Tipos corregidos")

# 4) NORMALIZACIÓN DE TEXTO
orders_clean.columns = orders_clean.columns.str.strip().str.lower().str.replace(' ', '_')
for col in orders_clean.select_dtypes(include=['object', 'string']).columns:
    orders_clean[col] = orders_clean[col].str.strip().str.lower().str.replace(' ', '_')
print("✅ Texto normalizado")

✅ Columnas seleccionadas
✅ Pedidos no entregados descartados: 2,963
✅ Tipos corregidos
✅ Texto normalizado


In [13]:
orders_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 96478 entries, 0 to 99440
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   order_id                  96478 non-null  string        
 1   customer_id               96478 non-null  string        
 2   order_purchase_timestamp  96478 non-null  datetime64[ns]
dtypes: datetime64[ns](1), string(2)
memory usage: 2.9 MB


In [14]:
## 5️⃣ GUARDAR
orders_clean.to_csv(os.path.join(output_path, "orders_clean.csv"), index=False)
print("💾 orders_clean.csv guardado")

💾 orders_clean.csv guardado


## 2.3 ORDER_ITEMS

In [15]:
## 1️⃣ DIMENSIÓN
print(f"Filas: {order_items_raw.shape[0]:,} | Columnas: {order_items_raw.shape[1]}")

Filas: 112,650 | Columnas: 7


In [16]:
## 2️⃣ INFORMACIÓN GENERAL
order_items_raw.info()
print("\nÚnicos:\n",   order_items_raw.nunique())
print("\nNulos:\n",    order_items_raw.isnull().any())
print("\nDuplicados:", order_items_raw.duplicated().any())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB

Únicos:
 order_id               98666
order_item_id             21
product_id             32951
seller_id               3095
shipping_limit_date    93318
price                   5968
freight_value           6999
dtype: int64

Nulos:
 order_id               False
order_item_id          False
product_id             False
seller_id              False
shipping_limit_dat

In [17]:
## 3️⃣ INSPECCIÓN
print("Primeras:\n",    order_items_raw.head(3))
print("\nÚltimas:\n",   order_items_raw.tail(3))

Primeras:
                            order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   

   shipping_limit_date  price  freight_value  
0  2017-09-19 09:45:35   58.9          13.29  
1  2017-05-03 11:05:13  239.9          19.93  
2  2018-01-18 14:48:30  199.0          17.87  

Últimas:
                                 order_id  order_item_id  \
112647  fffce4705a9662cd70adb13d4a31832d              1   
112648  fffe18544ffabc95dfada21779c9644f              1   
112649  fffe41c64501cc87c801fd61db3f6244              1   

                              product_id   

In [18]:
## 4️⃣ LIMPIEZA
order_items_clean = order_items_raw.copy()

# 1) SELECCIÓN DE COLUMNAS
order_items_clean = order_items_clean[['order_id', 'product_id', 'price']]
print("✅ Columnas seleccionadas")

# 2) CORRECCIÓN DE TIPOS
# los IDs son etiquetas, no números ni categorías — su tipo debe ser string
order_items_clean['order_id']   = order_items_clean['order_id'].astype('string')
order_items_clean['product_id'] = order_items_clean['product_id'].astype('string')
print("✅ Tipos corregidos")

# 3) NORMALIZACIÓN DE TEXTO
order_items_clean.columns = order_items_clean.columns.str.strip().str.lower().str.replace(' ', '_')
for col in order_items_clean.select_dtypes(include=['object', 'string']).columns:
    order_items_clean[col] = order_items_clean[col].str.strip().str.lower().str.replace(' ', '_')
print("✅ Texto normalizado")

✅ Columnas seleccionadas
✅ Tipos corregidos
✅ Texto normalizado


In [19]:
order_items_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   order_id    112650 non-null  string 
 1   product_id  112650 non-null  string 
 2   price       112650 non-null  float64
dtypes: float64(1), string(2)
memory usage: 2.6 MB


In [20]:
## 5️⃣ GUARDAR
order_items_clean.to_csv(os.path.join(output_path, "order_items_clean.csv"), index=False)
print("💾 order_items_clean.csv guardado")

💾 order_items_clean.csv guardado


## 2.4 PRODUCTS

In [21]:
## 1️⃣ DIMENSIÓN
print(f"Filas: {products_raw.shape[0]:,} | Columnas: {products_raw.shape[1]}")

Filas: 32,951 | Columnas: 9


In [22]:
## 2️⃣ INFORMACIÓN GENERAL
products_raw.info()
print("\nÚnicos:\n",    products_raw.nunique())
print("\nNulos (%):\n", products_raw.isnull().mean() * 100)
print("\nDuplicados:",  products_raw.duplicated().any())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB

Únicos:
 product_id                    32951
product_category_name            73
product_name_lenght              66
product_description_lenght     2960
product_photos_qty               19
product_weight_g       

In [23]:
## 3️⃣ INSPECCIÓN
print("Primeras:\n",    products_raw.head(3))
print("\nÚltimas:\n",   products_raw.tail(3))

Primeras:
                          product_id product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5            perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                 artes   
2  96bd76ec8810374ed1b65e291975717f         esporte_lazer   

   product_name_lenght  product_description_lenght  product_photos_qty  \
0                 40.0                       287.0                 1.0   
1                 44.0                       276.0                 1.0   
2                 46.0                       250.0                 1.0   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  
0             225.0               16.0               10.0              14.0  
1            1000.0               30.0               18.0              20.0  
2             154.0               18.0                9.0              15.0  

Últimas:
                              product_id   product_category_name  \
32948  9a7c6041fa9592d9d9ef6cfe62a71f8c         cama_mesa

In [24]:
## 4️⃣ LIMPIEZA
products_clean = products_raw.copy()

# 1) SELECCIÓN DE COLUMNAS
products_clean = products_clean[['product_id', 'product_category_name']]
print("✅ Columnas seleccionadas")

# 2) CORRECCIÓN DE TIPOS
# los IDs son etiquetas, no números ni categorías — su tipo debe ser string
products_clean['product_id'] = products_clean['product_id'].astype('string')
print("✅ Tipos corregidos")

# 3) NORMALIZACIÓN DE TEXTO
products_clean.columns = products_clean.columns.str.strip().str.lower().str.replace(' ', '_')
for col in products_clean.select_dtypes(include=['object', 'string']).columns:
    products_clean[col] = products_clean[col].str.strip().str.lower().str.replace(' ', '_')
print("✅ Texto normalizado")

# 4) GESTIÓN DE NULOS
# product_category_name tiene un 1.9% de nulos → imputamos con la moda
# variable categórica con < 5% de nulos → moda es la imputación más conservadora
moda_prod = products_clean['product_category_name'].mode()[0]
products_clean['product_category_name'] = products_clean['product_category_name'].fillna(moda_prod)
print(f"✅ Nulos imputados con la moda: '{moda_prod}'")

✅ Columnas seleccionadas
✅ Tipos corregidos
✅ Texto normalizado
✅ Nulos imputados con la moda: 'cama_mesa_banho'


In [25]:
products_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   product_id             32951 non-null  string
 1   product_category_name  32951 non-null  object
dtypes: object(1), string(1)
memory usage: 515.0+ KB


In [26]:
## 5️⃣ GUARDAR
products_clean.to_csv(os.path.join(output_path, "products_clean.csv"), index=False)
print("💾 products_clean.csv guardado")

💾 products_clean.csv guardado


## 2.5 REVIEWS

In [27]:
## 1️⃣ DIMENSIÓN
print(f"Filas: {reviews_raw.shape[0]:,} | Columnas: {reviews_raw.shape[1]}")

Filas: 99,224 | Columnas: 7


In [28]:
## 2️⃣ INFORMACIÓN GENERAL
reviews_raw.info()
print("\nÚnicos:\n",   reviews_raw.nunique())
print("\nNulos:\n",    reviews_raw.isnull().any())
print("\nDuplicados:", reviews_raw.duplicated().any())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB

Únicos:
 review_id                  98410
order_id                   98673
review_score                   5
review_comment_title        4527
review_comment_message     36159
review_creation_date         636
review_answer_timestamp    98248
dtype: int64

Nulos:
 review_id                  False
order_id                   False
review_score               False
rev

In [29]:
## 3️⃣ INSPECCIÓN
print("Primeras:\n",    reviews_raw.head(3))
print("\nÚltimas:\n",   reviews_raw.tail(3))

Primeras:
                           review_id                          order_id  \
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2  228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   

   review_score review_comment_title review_comment_message  \
0             4                  NaN                    NaN   
1             5                  NaN                    NaN   
2             5                  NaN                    NaN   

  review_creation_date review_answer_timestamp  
0  2018-01-18 00:00:00     2018-01-18 21:46:59  
1  2018-03-10 00:00:00     2018-03-11 03:05:13  
2  2018-02-17 00:00:00     2018-02-18 14:36:24  

Últimas:
                               review_id                          order_id  \
99221  b3de70c89b1510c4cd3d0649fd302472  55d4004744368f5571d1f590031933e4   
99222  1adeb9d84d72fe4e337617733eb85149  7725825d039fc1f0ceb7635e3f7d9206   
99223

In [30]:
## 4️⃣ LIMPIEZA
reviews_clean = reviews_raw.copy()

# 1) ORDENAR POR FECHA
# garantiza que keep='last' conserva la valoración más reciente, no la última por posición
reviews_clean = reviews_clean.sort_values('review_creation_date')
print("✅ Ordenado por fecha")

# 2) SELECCIÓN DE COLUMNAS
reviews_clean = reviews_clean[['order_id', 'review_score']]
print("✅ Columnas seleccionadas")

# 3) CORRECCIÓN DE TIPOS
reviews_clean['order_id'] = reviews_clean['order_id'].astype('string')
print("✅ Tipos corregidos")

# 4) DUPLICADOS POR order_id
# un pedido puede tener más de una valoración — conservamos la más reciente
antes = len(reviews_clean)
reviews_clean = reviews_clean.drop_duplicates(subset='order_id', keep='last')
print(f"✅ Valoraciones duplicadas eliminadas: {antes - len(reviews_clean):,}")

# 5) NORMALIZACIÓN DE TEXTO
reviews_clean.columns = reviews_clean.columns.str.strip().str.lower().str.replace(' ', '_')
for col in reviews_clean.select_dtypes(include=['object', 'string']).columns:
    reviews_clean[col] = reviews_clean[col].str.strip().str.lower().str.replace(' ', '_')
print("✅ Texto normalizado")

✅ Ordenado por fecha
✅ Columnas seleccionadas
✅ Tipos corregidos
✅ Valoraciones duplicadas eliminadas: 551
✅ Texto normalizado


In [31]:
reviews_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98673 entries, 70906 to 17900
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   order_id      98673 non-null  string
 1   review_score  98673 non-null  int64 
dtypes: int64(1), string(1)
memory usage: 2.3 MB


In [32]:
## 5️⃣ GUARDAR
reviews_clean.to_csv(os.path.join(output_path, "reviews_clean.csv"), index=False)
print("💾 reviews_clean.csv guardado")

💾 reviews_clean.csv guardado
